# Check for Augmentation / Near-Duplicates

Extracts ResNet50 features for every image (plus a horizontally-flipped pass), then flags image pairs above a cosine-similarity threshold to catch duplicates and flip augmentations. Writes a text report of the matches.

In [ ]:
import os
import numpy as np
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

DATASET_DIR       = r"D:\Desktop\Phyllody Infected - Gemini"
SIMILARITY_THRESHOLD = 0.98
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

transform_flipped = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

backbone = models.resnet50(weights='IMAGENET1K_V1')
backbone.fc = torch.nn.Identity()
backbone = backbone.to(DEVICE).eval()

files = [f for f in os.listdir(DATASET_DIR)
         if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"Extracting features from {len(files)} images...")
features      = []
features_flip = []
paths         = []

with torch.no_grad():
    for fname in tqdm(files):
        fpath = os.path.join(DATASET_DIR, fname)
        try:
            img       = Image.open(fpath).convert('RGB')
            feat      = backbone(transform(img).unsqueeze(0).to(DEVICE))
            feat_flip = backbone(transform_flipped(img).unsqueeze(0).to(DEVICE))
            features.append(feat.cpu().numpy()[0])
            features_flip.append(feat_flip.cpu().numpy()[0])
            paths.append(fpath)
        except Exception as e:
            print(f"  {fname}: {e}")

features      = np.array(features)
features_flip = np.array(features_flip)
print(f"Features extracted — shape: {features.shape}")

print("\nComputing similarity matrix...")
sim_matrix      = cosine_similarity(features)
sim_matrix_flip = cosine_similarity(features,
                                    features_flip)

print("Similarity computed")

similar_pairs = []

for i in tqdm(range(len(paths)), desc="Finding pairs"):
    for j in range(i + 1, len(paths)):
        sim_normal = sim_matrix[i, j]
        sim_flip   = sim_matrix_flip[i, j]
        best_sim   = max(sim_normal, sim_flip)

        if best_sim >= SIMILARITY_THRESHOLD:
            aug_type = "Horizontal Flip" if sim_flip > sim_normal else "Near Duplicate / Augmentation"
            similar_pairs.append({
                'path1':    paths[i],
                'path2':    paths[j],
                'file1':    os.path.basename(paths[i]),
                'file2':    os.path.basename(paths[j]),
                'sim':      round(float(best_sim), 4),
                'aug_type': aug_type
            })

similar_pairs.sort(key=lambda x: x['sim'], reverse=True)
print(f"\n Found {len(similar_pairs)} similar pairs above threshold {SIMILARITY_THRESHOLD}")

def show_pairs(pairs, n=100, title="Similar Pairs"):
    samples = pairs[:n]
    if not samples:
        print("No pairs to show.")
        return
    fig, axes = plt.subplots(n, 2, figsize=(8, n * 3))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    if n == 1: axes = [axes]
    for i, pair in enumerate(samples):
        img1 = np.array(Image.open(pair['path1']).convert('RGB'))
        img2 = np.array(Image.open(pair['path2']).convert('RGB'))
        h1, w1 = img1.shape[:2]
        h2, w2 = img2.shape[:2]

        axes[i][0].imshow(img1)
        axes[i][0].set_title(f"{pair['file1'][:30]}\n{w1}x{h1}px", fontsize=7)
        axes[i][0].axis('off')

        axes[i][1].imshow(img2)
        axes[i][1].set_title(
            f"{pair['file2'][:30]}\n{w2}x{h2}px | {pair['aug_type']} | sim={pair['sim']}",
            fontsize=7)
        axes[i][1].axis('off')

    plt.tight_layout()
    plt.show()

show_pairs(similar_pairs, n=min(100, len(similar_pairs)))

report_path = os.path.join(os.path.dirname(DATASET_DIR), "similarity_report.txt")
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(f"Similarity Report\nDataset: {DATASET_DIR}\n")
    f.write(f"Threshold: {SIMILARITY_THRESHOLD} | Pairs found: {len(similar_pairs)}\n\n")
    for p in similar_pairs:
        f.write(f"{p['aug_type']} | sim={p['sim']}\n")
        f.write(f"  {p['file1']}\n  {p['file2']}\n\n")

print(f"Report saved to {report_path}")